<a href="https://colab.research.google.com/github/mickymags/curriculum_development_initiative/blob/main/notebooks/Module_9_Intersection_Over_Union.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this module, we will calculate the intersection over union between flood maps. To do this, we will first need to export all of our flood maps to a common resolution.

# Step 1: Import packages

In [ ]:
import ee
import geemap
from google.colab import drive
import os
import glob
from osgeo import gdal
import numpy as np
import pandas as pd
import time

In [ ]:
ee.Authenticate()

ee.Initialize(project='servir-sco-assets')

# MODIFIABLE VARIABLE ALERT

In [ ]:
my_gee_folder = "users/mickymags/flood_intercomparison_chad_09_26_take2/"
my_Gdrive_folder = "/content/drive/MyDrive/Flood_Intercomparison/Case_Studies/confirmed_case_studies/cambodia_20241001/"
other_Gdrive_folder = "/content/drive/MyDrive/Flood_Intercomparison/Case_Studies/Flood_Intercomparison/"
flood_event_desc = 'chad_20240926'
time_id = '05280442'                # put in a string correlating to the current time which you are running the code -- will help with identifying exports

In [ ]:
aoi = ee.FeatureCollection(my_gee_folder + "aoi")
roi = aoi.geometry()
aoi_centroid = aoi.geometry().centroid()             # Get the center of the AOI
lon = aoi_centroid.coordinates().get(0).getInfo()    # Extract the longitude from the centroid
lat = aoi_centroid.coordinates().get(1).getInfo()    # Extract the latitude from the centroid

# Step 1: Import data from Google Earth Engine

In [ ]:
dswxhls = ee.Image(my_gee_folder + 'dswxhls_harmonized_30')
dswxs1 = ee.Image(my_gee_folder + 'dswxs1_harmonized_30')
gfm = ee.Image(my_gee_folder + 'gfm_harmonized')
hydrafloods = ee.Image(my_gee_folder + 'hydrafloods_harmonized_30')
hydrosar = ee.Image(my_gee_folder + 'hydrosar_harmonized')
mcdwd = ee.Image(my_gee_folder + 'mcdwd_harmonized_30')
vfm = ee.Image(my_gee_folder + 'vfm_harmonized_30')
aoi = ee.FeatureCollection(my_gee_folder + "aoi")

In [ ]:
drive.mount('/content/drive/')

In [ ]:
os.chdir(other_Gdrive_folder)

In [ ]:
pwd

'/content/drive/MyDrive/Flood_Intercomparison/Case_Studies/Flood_Intercomparison'

In [ ]:
#mydesc = a string of the time you submitted
def iou(img1, img2, desc1, desc2, aoi, myproj, mydesc):   #num_pixels
  img1_renamed = img1.rename(desc1)
  img2_renamed = img2.rename(desc2)

  combo = img1_renamed.addBands(img2_renamed)

  # Add a random number to the end of the export string
  #random = np.random.randint(1e7)
  #randstr = str(random)

  # Tile aoi
  covering_grid = aoi.geometry().coveringGrid(myproj, 2e5)

  num_tiles = covering_grid.size().getInfo()

  for j in range(num_tiles):
    my_tile = ee.Feature(covering_grid.toList(num_tiles).get(j))
    my_tile_clipped = my_tile.geometry().intersection(aoi, 0.001)
    tile_id = str(j)
    tile_export_string = 'iou_'+desc1 + '_and_' + desc2 + '_' + mydesc + 'pt' + tile_id

    sample = combo.sample(
      region= my_tile_clipped,
      scale=30,
      projection = myproj,
      numPixels = 1e13#num_pixels
    )

    #export_string = 'iou_'+desc1 + '_and_' + desc2 + '_' + randstr

    print('sampling...')

    geemap.ee_export_vector_to_drive(
        collection=sample,
        description= tile_export_string,
        fileFormat='CSV',
        folder='Flood_Intercomparison',
        selectors = [desc1, desc2]
    )

  print('sleeping...')
  time.sleep(60)
  print('still sleeping...')
  time.sleep(120)

  # For each tile

  dataframes = []
  combined_dataframe = pd.DataFrame()

  for k in range(num_tiles):
    #Read file in
    another_id = str(k)
    tile_input_string = 'iou_'+desc1 + '_and_' + desc2 + '_'  + mydesc + 'pt' + another_id
    data = pd.read_csv(tile_input_string + '.csv')
    combined_dataframe = pd.concat([combined_dataframe, data])
    #dataframes.append(data)
    ######
  #data = pd.read_csv(export_string + '.csv')

  intersection = 0
  union = 0

  print('calculating iou...')

  #num_rows = int(num_pixels - 1)
  num_rows = int(np.floor(combined_dataframe.size / 2))
  print(num_rows)

  for j in range(num_rows):
    row = combined_dataframe.iloc[j]
    feat1 = row[desc1]
    feat2 = row[desc2]

    perc = j * 100 / num_rows

    if j % 500000 == 0:
      print('iou calculation is {0:0.1f} % complete'.format(perc))

    if feat1 == 2 or feat2 == 2:
      continue
    if feat1 == 1 and feat2 == 1:
      intersection += 1
    if feat1 == 1 or feat2 == 1:
      union += 1

  return intersection/union

In [ ]:
my_proj_code = gfm.projection().getInfo()['crs']

In [ ]:
pwd

'/content/drive/MyDrive/Flood_Intercomparison/Case_Studies/confirmed_case_studies/cambodia_20241001'

In [ ]:
chad_gfm_hydrosar_iou = iou(gfm, hydrosar, 'gfm', 'hydrosar', aoi, my_proj_code, '06031110')

sampling...
Exporting iou_gfm_and_hydrosar_06031110pt0... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_gfm_and_hydrosar_06031110pt1... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_gfm_and_hydrosar_06031110pt2... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_gfm_and_hydrosar_06031110pt3... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_gfm_and_hydrosar_06031110pt4... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_gfm_and_hydrosar_06031110pt5... Please check the Task Manager from the JavaScript Code Editor.
sleeping...
still sleeping...
calculating iou...
38251216
iou calculation is 0.0 % complete
iou calculation is 1.3 % complete
iou calculation is 2.6 % complete
iou calculation is 3.9 % complete
iou calculation is 5.2 % complete
iou calculation is 6.5 % complete
iou calculat

In [ ]:
chad_gfm_hydrosar_iou

0.18841999080835117

In [ ]:
chad_gfm_hydrafloods_iou = iou(gfm, hydrafloods, 'gfm', 'hydrafloods', aoi, my_proj_code, time_id)

sampling...
Exporting iou_gfm_and_hydrafloods_05280442pt0... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_gfm_and_hydrafloods_05280442pt1... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_gfm_and_hydrafloods_05280442pt2... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_gfm_and_hydrafloods_05280442pt3... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_gfm_and_hydrafloods_05280442pt4... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_gfm_and_hydrafloods_05280442pt5... Please check the Task Manager from the JavaScript Code Editor.
sleeping...


KeyboardInterrupt: 

In [ ]:
chad_hydrafloods_hydrosar_iou = iou(hydrafloods, hydrosar, 'hydrafloods', 'hydrosar', aoi, my_proj_code, '06031148')

sampling...
Exporting iou_hydrafloods_and_hydrosar_06031148pt0... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_hydrafloods_and_hydrosar_06031148pt1... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_hydrafloods_and_hydrosar_06031148pt2... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_hydrafloods_and_hydrosar_06031148pt3... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_hydrafloods_and_hydrosar_06031148pt4... Please check the Task Manager from the JavaScript Code Editor.
sampling...
Exporting iou_hydrafloods_and_hydrosar_06031148pt5... Please check the Task Manager from the JavaScript Code Editor.
sleeping...
still sleeping...
calculating iou...
38251216
iou calculation is 0.0 % complete
iou calculation is 1.3 % complete
iou calculation is 2.6 % complete
iou calculation is 3.9 % complete
iou calculation is 5.2 % complet